In [ ]:
from faster_whisper import WhisperModel
import soundfile as sf
import numpy as np
import re, json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


# ---- Configuration ----
FILLERS = [
    "um", "uh", "uhh", "uhhh",
    "erm", "er", "eh", "ehh",
    "ah", "aah", "aaah",
    "hmm", "mmm", "mm",
    "like", "you know", "basically", "actually"
]

STAR_PATTERN = {
    "Situation": r"\b(Situation|context|when)\b",
    "Task": r"\b(Task|goal|objective)\b",
    "Action": r"\b(Action|I decided|I did)\b",
    "Result": r"\b(Result|outcome|impact)\b"
}

def compute_metrics(transcript, duration_s):
    words = transcript.split()
    n_words = len(words)
    wpm = (n_words / duration_s) * 60 if duration_s > 0 else 0

    filler_pattern = r"\b(" + "|".join(re.escape(f) for f in FILLERS) + r")+\b"
    filler_count = len(re.findall(filler_pattern, transcript.lower()))
    filler_pct = (filler_count / n_words) * 100 if n_words > 0 else 0

    return {
        "wpm": round(wpm, 1),
        "filler_pct": round(filler_pct, 1),
        "duration_s": round(duration_s, 1),
        "word_count": n_words,
    }

def rule_based_hints(metrics, transcript):
    hints = []
    if metrics["wpm"] > 170:
        hints.append("Pace > 170 WPM")
    if metrics["filler_pct"] > 5:
        hints.append("Filler > 5%")
    missing = [k for k, p in STAR_PATTERN.items() if not re.search(p, transcript, re.I)]
    if missing:
        hints.append(f"STAR: {', '.join(missing)} missing")
    return hints or ["Good delivery!"]

def transcribe(audio_path):
    model = WhisperModel("small", device="cpu")
    segments, info = model.transcribe(audio_path)
    transcript = " ".join([seg.text.strip() for seg in segments])
    metrics = compute_metrics(transcript, info.duration)
    hints = rule_based_hints(metrics, transcript)
    result = {"transcript": transcript, "metrics": metrics, "hints": hints}
    return result


In [ ]:
def explain_results(result):
    metrics = result["metrics"]
    hints = result["hints"]
    transcript = result["transcript"].lower()

    # same filler list as your main script
    FILLERS = ["um", "uh", "like", "you know", "aah", "mmm", "er", "eh", "hmm"]

    print("\n🧾 EXPLANATION:")

    # 1️⃣ WPM explanation
    if metrics["wpm"] < 100:
        print(f"- You spoke slowly ({metrics['wpm']} WPM). Try being a bit more energetic.")
    elif metrics["wpm"] > 150:
        print(f"- You spoke quite fast ({metrics['wpm']} WPM). Ideal range: 120–160 WPM.")
    else:
        print(f"- Your pace ({metrics['wpm']} WPM) is in a good range for interviews.")

    # 2️⃣ Filler explanation
    found_fillers = [f for f in FILLERS if f in transcript]

    if metrics["filler_pct"] > 10:
        print(f"- Fillers made up {metrics['filler_pct']}% of your words → shows nervousness. Practice silent pauses.")
    elif metrics["filler_pct"] > 5:
        print(f"- Fillers made up {metrics['filler_pct']}% of your words → mild hesitation, but manageable.")
    else:
        print(f"- Very few fillers ({metrics['filler_pct']}%). Great verbal control.")

    # 👉 Show which fillers were actually found
    if found_fillers:
        print(f"  • Fillers detected: {', '.join(found_fillers)}")
    else:
        print("  • No common fillers detected 👏")

    # 3️⃣ STAR explanation
    if any("STAR" in h for h in hints):
        print("- STAR structure incomplete — include Situation, Task, Action, and Result clearly.")
    else:
        print("- STAR structure complete — strong storytelling!")

    # 4️⃣ Overall summary
    if metrics["wpm"] > 170 or metrics["filler_pct"] > 10:
        print("⚠️ Overall: Confident but needs calmer delivery and fewer fillers.")
    elif any("STAR" in h for h in hints):
        print("🟡 Overall: Clear tone, but expand on your story with full STAR detail.")
    else:
        print("✅ Overall: Great delivery balance — clear, confident, and structured.")


In [8]:
import sounddevice as sd
from scipy.io.wavfile import write

fs = 16000  # Sample rate (Hz)
duration = 10  # seconds — you can increase to 20 or 30 if you like
output_path = "sample.wav"

print("🎤 Recording... start speaking now!")
recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
sd.wait()  # Wait until recording is finished
write(output_path, fs, recording)
print(f"✅ Saved recording as {output_path}")


🎤 Recording... start speaking now!
✅ Saved recording as sample.wav


In [ ]:
!conda install -c conda-forge ffmpeg soundfile -y


In [ ]:
import os
import subprocess
import soundfile as sf
from tkinter import Tk, filedialog

# --- 1️⃣ Open file picker window ---
Tk().withdraw()  # hide the empty tkinter window

file_path = filedialog.askopenfilename(
    title="🎵 Select an audio/video file",
    filetypes=[("Audio/Video Files", "*.mp3 *.mp4 *.m4a *.wav"), ("All Files", "*.*")]
)

if not file_path:
    print("⚠️ No file selected.")
else:
    print(f"📂 Selected file: {file_path}")

    # --- 2️⃣ Set output path (.wav) ---
    base, _ = os.path.splitext(file_path)
    wav_path = base + "_converted.wav"

    # --- 3️⃣ Convert to WAV (16kHz mono) using FFmpeg ---
    print("🎬 Converting to .wav ... please wait.")
    command = [
        "ffmpeg", "-y",
        "-i", file_path,
        "-ar", "16000",  # sample rate
        "-ac", "1",      # mono
        wav_path
    ]
    process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # --- 4️⃣ Check and display details ---
    if os.path.exists(wav_path):
        data, samplerate = sf.read(wav_path)
        duration = len(data) / samplerate
        print(f"✅ Done! Saved as: {wav_path}")
        print(f"🎵 Sample rate: {samplerate} Hz")
        print(f"⏱ Duration: {duration:.1f} seconds")
    else:
        print("❌ Conversion failed. Check that FFmpeg is installed and available.")


In [9]:
# 1️⃣ Transcribe your recorded sample
result = transcribe("sample.wav")

# 2️⃣ Print the structured JSON output
print(json.dumps(result, indent=2))

# 3️⃣ Explain the metrics and hints in plain language
explain_results(result)


{
  "transcript": "So the situation was that our client's project was like falling behind the schedule because of poor communication, but...",
  "metrics": {
    "wpm": 114.0,
    "filler_pct": 5.3,
    "duration_s": 10.0,
    "word_count": 19
  },
  "hints": [
    "Filler > 5%",
    "STAR: Task, Action, Result missing"
  ]
}

🧾 EXPLANATION:
- You spoke slowly (114.0 WPM). Try being a bit more energetic.
- Fillers made up 5.3% of your words → mild hesitation, but manageable.
  • Fillers detected: like, eh
- STAR structure incomplete — include Situation, Task, Action, and Result clearly.
🟡 Overall: Clear tone, but expand on your story with full STAR detail.
